# SQL for Data Analytics — Companion Workbook

**Kajola Gbenga**
*Data Scientist & AI Engineer*

This notebook is the executable companion to the *SQL for Data Analytics* teaching note. It builds a small, realistic SQLite database in memory and runs real, executable SQL for the core techniques taught in the note: KPIs, sales analytics, profitability, customer analytics, RFM, cohorts, churn, market basket analysis, z-scores, Pareto analysis, growth and window functions, session analysis, and a leakage-safe machine learning feature table.

SQLite is used because it requires no server setup and ships with Python. Every query below is written in portable, standard SQL. Where a technique relies on a feature unique to another engine (for example, `PERCENTILE_CONT` in PostgreSQL/Oracle, or a native `CORR()` function), a note explains the difference and how it is approximated here.

Run the cells in order from top to bottom.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 140)

conn = sqlite3.connect(':memory:')

def run(sql, params=None):
    """Run a SQL query and return the result as a pandas DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)

print("SQLite version:", sqlite3.sqlite_version)
print("Connection ready.")

SQLite version: 3.45.1
Connection ready.


## 1. Build the Reference Schema and Synthetic Data

This mirrors the reference e-commerce schema used throughout the teaching note: `customers`, `products`, `orders`, `order_items`, `support_tickets`, and `website_events`. Data is generated synthetically but with realistic patterns (repeat customers, seasonal variation, a handful of high-value customers, some declining products) so the analytical queries below produce meaningful, non-trivial results.

In [2]:
conn.executescript('''
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS support_tickets;
DROP TABLE IF EXISTS website_events;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    first_name TEXT,
    last_name TEXT,
    email TEXT,
    signup_date TEXT,
    country TEXT,
    city TEXT,
    segment TEXT
);

CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    product_name TEXT,
    category TEXT,
    sub_category TEXT,
    unit_cost REAL,
    unit_price REAL
);

CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date TEXT,
    order_status TEXT,
    channel TEXT,
    region TEXT
);

CREATE TABLE order_items (
    order_item_id INTEGER PRIMARY KEY,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    unit_price REAL,
    discount REAL
);

CREATE TABLE support_tickets (
    ticket_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    opened_date TEXT,
    category TEXT,
    status TEXT
);

CREATE TABLE website_events (
    event_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    session_hint INTEGER,
    event_name TEXT,
    event_timestamp TEXT,
    device TEXT
);
''')
print("Schema created.")

Schema created.


In [3]:
rng = np.random.default_rng(42)

# --- customers ---
n_customers = 400
countries = ['Nigeria', 'United States', 'United Kingdom', 'Kenya', 'Ghana', 'South Africa']
segments = ['Consumer', 'Small Business', 'Enterprise']
cities = {'Nigeria': ['Lagos','Abuja','Ibadan'], 'United States': ['New York','Austin','Chicago'],
          'United Kingdom': ['London','Manchester'], 'Kenya': ['Nairobi'], 'Ghana': ['Accra'],
          'South Africa': ['Johannesburg','Cape Town']}

start_signup = datetime(2024, 1, 1)
customers = []
for cid in range(1, n_customers + 1):
    country = rng.choice(countries, p=[0.35,0.25,0.15,0.1,0.05,0.10])
    signup_offset = int(rng.integers(0, 620))
    customers.append((
        cid, f"First{cid}", f"Last{cid}", f"customer{cid}@example.com",
        (start_signup + timedelta(days=signup_offset)).strftime('%Y-%m-%d'),
        country, rng.choice(cities[country]), rng.choice(segments, p=[0.7,0.2,0.1])
    ))

conn.executemany("INSERT INTO customers VALUES (?,?,?,?,?,?,?,?)", customers)

# --- products ---
categories = {
    'Electronics': ['Headphones','Smartwatch','Bluetooth Speaker','Laptop Stand','Webcam'],
    'Home': ['Blender','Air Fryer','Desk Lamp','Coffee Maker'],
    'Apparel': ['T-Shirt','Jacket','Sneakers','Cap'],
    'Office': ['Notebook Set','Ergonomic Chair','Desk Organizer'],
}
products = []
pid = 1
for cat, items in categories.items():
    for item in items:
        cost = round(rng.uniform(5, 200), 2)
        price = round(cost * rng.uniform(1.4, 2.6), 2)
        products.append((pid, item, cat, item, cost, price))
        pid += 1

conn.executemany("INSERT INTO products VALUES (?,?,?,?,?,?)", products)
n_products = len(products)
print(f"{n_customers} customers, {n_products} products inserted.")

400 customers, 16 products inserted.


In [4]:
# --- orders and order_items: repeat purchase behavior, seasonality, a few VIPs ---
order_id = 1
item_id = 1
orders_rows = []
items_rows = []

channels = ['Online', 'Mobile App', 'In-Store']
regions = ['North', 'South', 'East', 'West']
statuses_pool = ['completed']*92 + ['cancelled']*5 + ['refunded']*3

start_date = datetime(2024, 1, 1)
end_date = datetime(2026, 8, 31)

# Give each customer a "purchase intensity" so behavior is realistic (some VIPs, many one-timers)
intensity = rng.gamma(shape=1.3, scale=2.2, size=n_customers)  # avg ~3 orders, long tail

for idx, cust in enumerate(customers):
    cid = cust[0]
    signup = datetime.strptime(cust[4], '%Y-%m-%d')
    n_orders = max(0, int(rng.poisson(intensity[idx])))
    last_date = signup
    for _ in range(n_orders):
        gap = int(rng.exponential(45)) + 1
        order_date = last_date + timedelta(days=gap)
        if order_date > end_date:
            break
        last_date = order_date
        status = rng.choice(statuses_pool)
        orders_rows.append((
            order_id, cid, order_date.strftime('%Y-%m-%d'), status,
            rng.choice(channels, p=[0.55,0.30,0.15]), rng.choice(regions)
        ))
        n_lines = int(rng.integers(1, 4))
        chosen_products = rng.choice(range(1, n_products+1), size=n_lines, replace=False)
        for prod_idx in chosen_products:
            prod = products[prod_idx - 1]
            qty = int(rng.integers(1, 5))
            discount = float(rng.choice([0, 0, 0, 0.05, 0.1, 0.15], p=[0.55,0.1,0.1,0.1,0.1,0.05]))
            items_rows.append((item_id, order_id, prod[0], qty, prod[5], discount))
            item_id += 1
        order_id += 1

conn.executemany("INSERT INTO orders VALUES (?,?,?,?,?,?)", orders_rows)
conn.executemany("INSERT INTO order_items VALUES (?,?,?,?,?,?)", items_rows)
conn.commit()
print(f"{len(orders_rows)} orders, {len(items_rows)} order line items inserted.")

1122 orders, 2261 order line items inserted.


In [5]:
# --- support tickets (sparse) ---
tickets = []
tid = 1
for cust in customers:
    if rng.random() < 0.25:
        n_tix = int(rng.integers(1, 3))
        for _ in range(n_tix):
            signup = datetime.strptime(cust[4], '%Y-%m-%d')
            offset = int(rng.integers(0, 500))
            tdate = signup + timedelta(days=offset)
            if tdate > end_date:
                continue
            tickets.append((tid, cust[0], tdate.strftime('%Y-%m-%d'),
                             rng.choice(['Billing','Shipping','Product','Account']),
                             rng.choice(['closed','closed','open'])))
            tid += 1

conn.executemany("INSERT INTO support_tickets VALUES (?,?,?,?,?)", tickets)

# --- website events (for funnel / DAU / session examples) ---
events = []
eid = 1
event_sequence = ['visited','viewed_product','added_to_cart','started_checkout','purchased']
for cust in customers:
    if rng.random() < 0.6:
        n_sessions = int(rng.integers(1, 6))
        signup = datetime.strptime(cust[4], '%Y-%m-%d')
        for s in range(n_sessions):
            offset_days = int(rng.integers(0, 550))
            base_time = signup + timedelta(days=offset_days, hours=int(rng.integers(8,22)))
            depth = rng.choice([1,2,3,4,5], p=[0.30,0.25,0.20,0.15,0.10])
            device = rng.choice(['mobile','desktop'], p=[0.6,0.4])
            for step in range(depth):
                events.append((eid, cust[0], s, event_sequence[step],
                                (base_time + timedelta(minutes=step*3)).strftime('%Y-%m-%d %H:%M:%S'), device))
                eid += 1

conn.executemany("INSERT INTO website_events VALUES (?,?,?,?,?,?)", events)
conn.commit()
print(f"{len(tickets)} support tickets, {len(events)} website events inserted.")

145 support tickets, 1899 website events inserted.


## 2. Data Exploration and Profiling (Part 4)

Before any analysis, profile the tables: row counts, missing values, duplicates, and basic distributions.

In [6]:
display(run("""
SELECT
    (SELECT COUNT(*) FROM customers) AS customers,
    (SELECT COUNT(*) FROM products) AS products,
    (SELECT COUNT(*) FROM orders) AS orders,
    (SELECT COUNT(*) FROM order_items) AS order_items;
"""))

,customers,products,orders,order_items
0,400,16,1122,2261


In [7]:
# Missing values and basic distribution checks
display(run("""
SELECT
    SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS missing_customer_id,
    SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS missing_order_date,
    COUNT(DISTINCT order_status) AS distinct_statuses
FROM orders;
"""))

display(run("""
SELECT order_status, COUNT(*) AS orders,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM orders), 1) AS pct_of_total
FROM orders GROUP BY order_status ORDER BY orders DESC;
"""))

,missing_customer_id,missing_order_date,distinct_statuses
0,0,0,3


,order_status,orders,pct_of_total
0,completed,1038,92.5
1,cancelled,54,4.8
2,refunded,30,2.7


## 3. KPI Development (Part 8): Revenue, AOV, Margin

Revenue is always computed net of discount, at the order-item grain, then aggregated (Part 3: grain matters).

In [8]:
kpis = run("""
SELECT
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS revenue,
    ROUND(SUM(oi.quantity * p.unit_cost), 2) AS cost,
    ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) - SUM(oi.quantity * p.unit_cost), 2) AS gross_profit,
    ROUND(100.0 * (SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) - SUM(oi.quantity * p.unit_cost))
          / SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 1) AS gross_margin_pct,
    COUNT(DISTINCT o.order_id) AS orders,
    COUNT(DISTINCT o.customer_id) AS customers
FROM order_items oi
JOIN orders o ON o.order_id = oi.order_id
JOIN products p ON p.product_id = oi.product_id
WHERE o.order_status = 'completed';
""")
display(kpis)

aov = run("""
WITH order_revenue AS (
    SELECT o.order_id, SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) AS order_total
    FROM order_items oi JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_status = 'completed'
    GROUP BY o.order_id
)
SELECT ROUND(AVG(order_total), 2) AS average_order_value FROM order_revenue;
""")
display(aov)

,revenue,cost,gross_profit,gross_margin_pct,orders,customers
0,974191.43,498426.18,475765.25,48.8,1038,297


,average_order_value
0,938.53


## 4. Sales Analytics (Part 9): Monthly Revenue, MoM and YoY Growth

SQLite has no native `DATE_TRUNC`; `strftime('%Y-%m', order_date)` achieves the same month-level grouping.

In [9]:
monthly = run("""
SELECT strftime('%Y-%m', o.order_date) AS month,
       ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS revenue
FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.order_status = 'completed'
GROUP BY 1 ORDER BY 1;
""")

monthly['prior_month_revenue'] = monthly['revenue'].shift(1)
monthly['mom_growth_pct'] = ((monthly['revenue'] - monthly['prior_month_revenue']) / monthly['prior_month_revenue'] * 100).round(1)
display(monthly.tail(12))

,month,revenue,prior_month_revenue,mom_growth_pct
20,2025-09,35051.63,38777.17,-9.6
21,2025-10,40744.26,35051.63,16.2
22,2025-11,37391.01,40744.26,-8.2
23,2025-12,34098.97,37391.01,-8.8
24,2026-01,24745.48,34098.97,-27.4
25,2026-02,8375.43,24745.48,-66.2
26,2026-03,9607.32,8375.43,14.7
27,2026-04,11029.30,9607.32,14.8
28,2026-05,14897.80,11029.30,35.1
29,2026-06,11124.40,14897.80,-25.3


In [10]:
# Same growth calculation done entirely in SQL using a window LAG (Part 43)
growth_sql = run("""
WITH monthly AS (
    SELECT strftime('%Y-%m', o.order_date) AS month,
           SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) AS revenue
    FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.order_status = 'completed'
    GROUP BY 1
)
SELECT month, ROUND(revenue,2) AS revenue,
       ROUND(LAG(revenue) OVER (ORDER BY month), 2) AS prior_month_revenue,
       ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY month)) / LAG(revenue) OVER (ORDER BY month), 1) AS mom_growth_pct
FROM monthly ORDER BY month;
""")
display(growth_sql.tail(12))

,month,revenue,prior_month_revenue,mom_growth_pct
20,2025-09,35051.63,38777.17,-9.6
21,2025-10,40744.26,35051.63,16.2
22,2025-11,37391.01,40744.26,-8.2
23,2025-12,34098.97,37391.01,-8.8
24,2026-01,24745.48,34098.97,-27.4
25,2026-02,8375.43,24745.48,-66.2
26,2026-03,9607.32,8375.43,14.7
27,2026-04,11029.30,9607.32,14.8
28,2026-05,14897.80,11029.30,35.1
29,2026-06,11124.40,14897.80,-25.3


## 5. Sales Breakdown by Region, Category, and Channel (Part 9)

In [11]:
display(run("""
SELECT o.region, ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount)),2) AS revenue
FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.order_status='completed'
GROUP BY o.region ORDER BY revenue DESC;
"""))

display(run("""
SELECT p.category, ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount)),2) AS revenue,
       ROUND(100.0*SUM(oi.quantity*oi.unit_price*(1-oi.discount)) /
             (SELECT SUM(quantity*unit_price*(1-discount)) FROM order_items), 1) AS pct_of_total
FROM order_items oi JOIN products p ON p.product_id = oi.product_id
GROUP BY p.category ORDER BY revenue DESC;
"""))

,region,revenue
0,East,252196.39
1,South,251976.25
2,West,236619.14
3,North,233399.65


,category,revenue,pct_of_total
0,Electronics,330335.27,31.2
1,Office,309880.84,29.3
2,Home,264938.13,25.0
3,Apparel,152725.50,14.4


## 6. Profitability Analysis (Part 10): Most and Least Profitable Products

In [12]:
profit = run("""
SELECT p.product_id, p.product_name,
       ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount)),2) AS revenue,
       ROUND(SUM(oi.quantity*p.unit_cost),2) AS cost,
       ROUND(SUM(oi.quantity*oi.unit_price*(1-oi.discount)) - SUM(oi.quantity*p.unit_cost),2) AS gross_profit,
       ROUND(100.0*(SUM(oi.quantity*oi.unit_price*(1-oi.discount)) - SUM(oi.quantity*p.unit_cost))
             / SUM(oi.quantity*oi.unit_price*(1-oi.discount)),1) AS margin_pct
FROM order_items oi JOIN products p ON p.product_id = oi.product_id
GROUP BY p.product_id, p.product_name
ORDER BY gross_profit DESC;
""")
print("Top 5 most profitable products:")
display(profit.head(5))
print("Bottom 5 least profitable products:")
display(profit.tail(5))

Top 5 most profitable products:
Bottom 5 least profitable products:


,product_id,product_name,revenue,cost,gross_profit,margin_pct
0,15,Ergonomic Chair,150762.53,59717.60,91044.93,60.4
1,2,Smartwatch,109280.75,54452.79,54827.96,50.2
2,12,Sneakers,80401.50,31595.22,48806.28,60.7
3,6,Blender,85387.65,37914.77,47472.88,55.6
4,4,Laptop Stand,68110.64,30392.32,37718.32,55.4


,product_id,product_name,revenue,cost,gross_profit,margin_pct
11,5,Webcam,37738.96,16725.60,21013.36,55.7
12,1,Headphones,28476.23,12043.12,16433.11,57.7
13,13,Cap,23367.28,10988.96,12378.32,53.0
14,8,Desk Lamp,25304.86,15563.24,9741.62,38.5
15,10,T-Shirt,12179.78,7728.35,4451.43,36.5


## 7. Customer Analytics and RFM Segmentation (Parts 11-13)

Recency, Frequency, and Monetary value, scored into quintiles with `NTILE(5)`, then mapped to named marketing segments exactly as in Part 13 of the teaching note.

In [13]:
rfm = run("""
WITH rfm_base AS (
    SELECT c.customer_id,
           CAST(julianday('2026-09-07') - julianday(MAX(o.order_date)) AS INTEGER) AS recency_days,
           COUNT(DISTINCT o.order_id) AS frequency,
           SUM(oi.quantity*oi.unit_price*(1-oi.discount)) AS monetary
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id AND o.order_status = 'completed'
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY c.customer_id
),
rfm_scored AS (
    SELECT customer_id, recency_days, frequency, monetary,
           NTILE(5) OVER (ORDER BY recency_days DESC) AS r_score,
           NTILE(5) OVER (ORDER BY frequency ASC) AS f_score,
           NTILE(5) OVER (ORDER BY monetary ASC) AS m_score
    FROM rfm_base
)
SELECT customer_id, recency_days, frequency, ROUND(monetary,2) AS monetary,
       r_score, f_score, m_score,
       CASE
           WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
           WHEN r_score >= 3 AND f_score >= 3 THEN 'Loyal Customers'
           WHEN r_score >= 4 AND f_score <= 2 THEN 'Potential Loyalists'
           WHEN r_score >= 4 AND f_score = 1 AND m_score = 1 THEN 'New Customers'
           WHEN r_score <= 2 AND f_score >= 3 THEN 'At Risk'
           WHEN r_score = 1 AND f_score >= 4 THEN 'Cannot Lose Them'
           WHEN r_score <= 2 AND f_score <= 2 AND m_score >= 3 THEN 'Hibernating'
           ELSE 'Lost'
       END AS rfm_segment
FROM rfm_scored;
""")
display(rfm.head(10))

print("\nRevenue and customer count by RFM segment:")
display(rfm.groupby('rfm_segment').agg(customers=('customer_id','count'), total_monetary=('monetary','sum'))
        .sort_values('total_monetary', ascending=False).round(2))


Revenue and customer count by RFM segment:


,customer_id,recency_days,frequency,monetary,r_score,f_score,m_score,rfm_segment
0,329,953,1,1304.80,1,2,2,Lost
1,94,950,1,1571.67,1,2,2,Lost
2,95,940,1,825.16,1,1,1,Lost
3,285,932,1,1203.15,1,1,2,Lost
4,310,916,1,1456.60,1,2,2,Lost
5,142,911,1,266.70,1,1,1,Lost
6,265,903,1,531.94,1,1,1,Lost
7,263,899,3,4080.11,1,4,4,At Risk
8,149,897,3,3664.62,1,3,4,At Risk
9,21,892,2,922.62,1,2,1,Lost


,customers,total_monetary
rfm_segment,,
Champions,58,409649.54
Loyal Customers,61,216349.37
At Risk,58,215956.30
Lost,83,87007.29
Potential Loyalists,33,35895.06
Hibernating,4,9333.79


## 8. Cohort Retention Analysis (Part 15)

Each customer's acquisition month is their cohort. `month_number` measures how many months after acquisition an activity month falls. The result below is the long-format version of the classic wide cohort retention table shown in the teaching note; pivoting it wide is one line in pandas.

In [14]:
cohort_long = run("""
WITH first_orders AS (
    SELECT customer_id, strftime('%Y-%m', MIN(order_date)) AS cohort_month
    FROM orders WHERE order_status='completed' GROUP BY customer_id
),
activity AS (
    SELECT o.customer_id, f.cohort_month, strftime('%Y-%m', o.order_date) AS activity_month
    FROM orders o JOIN first_orders f ON f.customer_id = o.customer_id
    WHERE o.order_status = 'completed'
),
cohort_index AS (
    SELECT customer_id, cohort_month, activity_month,
           (CAST(strftime('%Y', activity_month || '-01') AS INTEGER) - CAST(strftime('%Y', cohort_month || '-01') AS INTEGER)) * 12
         + (CAST(strftime('%m', activity_month || '-01') AS INTEGER) - CAST(strftime('%m', cohort_month || '-01') AS INTEGER)) AS month_number
    FROM activity
),
cohort_size AS (
    SELECT cohort_month, COUNT(DISTINCT customer_id) AS cohort_customers FROM first_orders GROUP BY cohort_month
)
SELECT ci.cohort_month, ci.month_number,
       COUNT(DISTINCT ci.customer_id) AS active_customers,
       cs.cohort_customers,
       ROUND(100.0 * COUNT(DISTINCT ci.customer_id) / cs.cohort_customers, 1) AS retention_pct
FROM cohort_index ci JOIN cohort_size cs ON cs.cohort_month = ci.cohort_month
GROUP BY ci.cohort_month, ci.month_number, cs.cohort_customers
ORDER BY ci.cohort_month, ci.month_number;
""")
display(cohort_long.head(15))

,cohort_month,month_number,active_customers,cohort_customers,retention_pct
0,2024-01,0,6,6,100.0
1,2024-01,1,2,6,33.3
2,2024-01,3,3,6,50.0
3,2024-01,4,2,6,33.3
4,2024-01,6,1,6,16.7
5,2024-01,7,1,6,16.7
6,2024-01,9,1,6,16.7
7,2024-02,0,10,10,100.0
8,2024-02,1,4,10,40.0
9,2024-02,2,2,10,20.0


In [15]:
# Pivot to the classic wide cohort table (first 6 months), done in pandas
cohort_wide = cohort_long[cohort_long['month_number'].between(0, 5)].pivot(
    index='cohort_month', columns='month_number', values='retention_pct'
)
cohort_wide.columns = [f"Month {c}" for c in cohort_wide.columns]
display(cohort_wide.head(10))

,Month 0,Month 1,Month 2,Month 3,Month 4,Month 5
cohort_month,,,,,,
2024-01,100.0,33.3,NaN,50.0,33.3,NaN
2024-02,100.0,40.0,20.0,40.0,40.0,20.0
2024-03,100.0,33.3,16.7,16.7,8.3,8.3
2024-04,100.0,41.7,25.0,16.7,8.3,16.7
2024-05,100.0,25.0,31.3,25.0,12.5,25.0
2024-06,100.0,42.9,28.6,21.4,7.1,14.3
2024-07,100.0,23.5,29.4,17.6,29.4,11.8
2024-08,100.0,57.9,15.8,31.6,10.5,15.8
2024-09,100.0,31.6,36.8,21.1,10.5,15.8


## 9. Churn Analytics (Part 16)

Churn is defined here as no completed order within 90 days of "today" (using 2026-09-07 as the analysis date, matching this workbook's reference date).

In [16]:
churn = run("""
WITH last_orders AS (
    SELECT customer_id, MAX(order_date) AS last_order_date
    FROM orders WHERE order_status = 'completed' GROUP BY customer_id
)
SELECT customer_id, last_order_date,
       CAST(julianday('2026-09-07') - julianday(last_order_date) AS INTEGER) AS days_since_last_order,
       CASE WHEN julianday('2026-09-07') - julianday(last_order_date) > 90 THEN 1 ELSE 0 END AS is_churned
FROM last_orders;
""")
churn_rate = churn['is_churned'].mean() * 100
print(f"Overall churn rate (90-day definition): {churn_rate:.1f}%")
display(churn.head())

Overall churn rate (90-day definition): 94.6%


,customer_id,last_order_date,days_since_last_order,is_churned
0,3,2024-06-10,819,1
1,4,2026-06-04,95,1
2,7,2025-09-03,369,1
3,9,2025-03-11,545,1
4,10,2025-02-13,571,1


In [17]:
# Churn rate by customer segment
churn_by_segment = run("""
WITH last_orders AS (
    SELECT customer_id, MAX(order_date) AS last_order_date
    FROM orders WHERE order_status = 'completed' GROUP BY customer_id
),
flagged AS (
    SELECT customer_id,
           CASE WHEN julianday('2026-09-07') - julianday(last_order_date) > 90 THEN 1 ELSE 0 END AS is_churned
    FROM last_orders
)
SELECT c.segment, ROUND(100.0 * AVG(f.is_churned), 1) AS churn_rate_pct, COUNT(*) AS customers
FROM flagged f JOIN customers c ON c.customer_id = f.customer_id
GROUP BY c.segment ORDER BY churn_rate_pct DESC;
""")
display(churn_by_segment)

,segment,churn_rate_pct,customers
0,Consumer,96.6,207
1,Small Business,91.4,58
2,Enterprise,87.5,32


## 10. Market Basket Analysis (Part 18): Products Frequently Bought Together

A self-join on shared `order_id` finds product pairs; support and confidence quantify how strong the association is.

In [18]:
basket = run("""
WITH order_totals AS (SELECT COUNT(DISTINCT order_id) AS n FROM orders WHERE order_status='completed'),
pair_counts AS (
    SELECT a.product_id AS product_a, b.product_id AS product_b, COUNT(DISTINCT a.order_id) AS both_count
    FROM order_items a
    JOIN order_items b ON a.order_id = b.order_id AND a.product_id < b.product_id
    JOIN orders o ON o.order_id = a.order_id AND o.order_status = 'completed'
    GROUP BY a.product_id, b.product_id
),
single_counts AS (
    SELECT oi.product_id, COUNT(DISTINCT oi.order_id) AS a_count
    FROM order_items oi JOIN orders o ON o.order_id = oi.order_id
    WHERE o.order_status = 'completed'
    GROUP BY oi.product_id
)
SELECT pa.product_name AS product_a, pb.product_name AS product_b,
       pc.both_count,
       ROUND(1.0 * pc.both_count / ot.n, 4) AS support,
       ROUND(1.0 * pc.both_count / sc.a_count, 4) AS confidence
FROM pair_counts pc
CROSS JOIN order_totals ot
JOIN single_counts sc ON sc.product_id = pc.product_a
JOIN products pa ON pa.product_id = pc.product_a
JOIN products pb ON pb.product_id = pc.product_b
ORDER BY pc.both_count DESC
LIMIT 15;
""")
display(basket)

,product_a,product_b,both_count,support,confidence
0,Laptop Stand,Jacket,22,0.0212,0.1705
1,Blender,T-Shirt,21,0.0202,0.1591
2,Headphones,Coffee Maker,20,0.0193,0.1575
3,Webcam,Notebook Set,19,0.0183,0.1610
4,Headphones,Ergonomic Chair,18,0.0173,0.1417
5,Smartwatch,Laptop Stand,18,0.0173,0.1429
6,Bluetooth Speaker,Coffee Maker,18,0.0173,0.1385
7,Blender,Air Fryer,18,0.0173,0.1364
8,Laptop Stand,Webcam,17,0.0164,0.1318
9,Laptop Stand,Blender,17,0.0164,0.1318


## 11. Z-Score Outlier Detection (Part 32)

Flag orders whose total is an unusual outlier relative to the overall order-value distribution.

In [19]:
orders_totals = run("""
SELECT o.order_id, o.customer_id,
       SUM(oi.quantity*oi.unit_price*(1-oi.discount)) AS order_total
FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
WHERE o.order_status = 'completed'
GROUP BY o.order_id, o.customer_id;
""")

mean_val = orders_totals['order_total'].mean()
std_val = orders_totals['order_total'].std()
orders_totals['z_score'] = (orders_totals['order_total'] - mean_val) / std_val

outliers = orders_totals[orders_totals['z_score'].abs() > 3].sort_values('z_score', ascending=False)
print(f"Mean order total: {mean_val:.2f}, Std Dev: {std_val:.2f}")
print(f"Orders flagged with |z| > 3: {len(outliers)}")
display(outliers.head(10))

Mean order total: 938.53, Std Dev: 613.29
Orders flagged with |z| > 3: 12


,order_id,customer_id,order_total,z_score
263,285,93,3323.320,3.888527
621,665,219,3192.488,3.675199
834,896,301,3149.690,3.605414
365,392,127,3073.920,3.481867
213,232,80,2962.650,3.300436
698,748,249,2955.080,3.288093
983,1062,365,2914.220,3.221468
22,27,15,2826.120,3.077817
773,828,279,2826.120,3.077817
395,422,141,2819.700,3.067349


In [20]:
# Equivalent z-score computation done entirely in SQL, using SQLite's window AVG()/statistical aggregates
zscore_sql = run("""
WITH order_totals AS (
    SELECT o.order_id, SUM(oi.quantity*oi.unit_price*(1-oi.discount)) AS order_total
    FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.order_status = 'completed'
    GROUP BY o.order_id
),
stats AS (
    SELECT AVG(order_total) AS mean_val,
           -- SQLite has no native STDDEV; compute variance manually, then take the square root in Python/pandas below
           AVG(order_total * order_total) - AVG(order_total) * AVG(order_total) AS variance_val
    FROM order_totals
)
SELECT ot.order_id, ot.order_total, s.mean_val, s.variance_val
FROM order_totals ot CROSS JOIN stats s
ORDER BY ot.order_total DESC LIMIT 5;
""")
zscore_sql['stddev_val'] = np.sqrt(zscore_sql['variance_val'])
zscore_sql['z_score'] = (zscore_sql['order_total'] - zscore_sql['mean_val']) / zscore_sql['stddev_val']
display(zscore_sql[['order_id','order_total','z_score']])
print("Note: PostgreSQL, MySQL 8+, SQL Server, and Oracle all provide native STDDEV()/STDDEV_POP() -")
print("SQLite is the exception among the five dialects covered in the teaching note, hence the manual variance formula above.")

Note: PostgreSQL, MySQL 8+, SQL Server, and Oracle all provide native STDDEV()/STDDEV_POP() -
SQLite is the exception among the five dialects covered in the teaching note, hence the manual variance formula above.


,order_id,order_total,z_score
0,285,3323.320,3.890401
1,665,3192.488,3.676970
2,896,3149.690,3.607152
3,392,3073.920,3.483546
4,232,2962.650,3.302027


## 12. Pareto (80/20) Analysis (Part 41)

What percentage of customers generate 80% of total revenue?

In [21]:
pareto = run("""
WITH customer_revenue AS (
    SELECT c.customer_id, SUM(oi.quantity*oi.unit_price*(1-oi.discount)) AS revenue
    FROM customers c
    JOIN orders o ON o.customer_id = c.customer_id AND o.order_status='completed'
    JOIN order_items oi ON oi.order_id = o.order_id
    GROUP BY c.customer_id
),
ranked AS (
    SELECT customer_id, revenue,
           SUM(revenue) OVER (ORDER BY revenue DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_revenue,
           SUM(revenue) OVER () AS total_revenue,
           ROW_NUMBER() OVER (ORDER BY revenue DESC) AS rn,
           COUNT(*) OVER () AS total_customers
    FROM customer_revenue
)
SELECT customer_id, ROUND(revenue,2) AS revenue,
       ROUND(100.0 * running_revenue / total_revenue, 1) AS cumulative_revenue_pct,
       ROUND(100.0 * rn / total_customers, 1) AS cumulative_customer_pct
FROM ranked ORDER BY rn;
""")

threshold_row = pareto[pareto['cumulative_revenue_pct'] >= 80].iloc[0]
print(f"{threshold_row['cumulative_customer_pct']}% of customers generate 80% of total revenue.")
display(pareto.head(10))

48.1% of customers generate 80% of total revenue.


,customer_id,revenue,cumulative_revenue_pct,cumulative_customer_pct
0,80,17753.26,1.8,0.3
1,264,15540.51,3.4,0.7
2,217,13147.36,4.8,1.0
3,25,12995.94,6.1,1.3
4,141,12102.88,7.3,1.7
5,66,11852.77,8.6,2.0
6,117,11697.73,9.8,2.4
7,130,11091.21,10.9,2.7
8,146,10986.28,12.0,3.0
9,93,10720.83,13.1,3.4


## 13. Session Construction from Event Logs (Part 38)

Group raw `website_events` into sessions using a 30-minute inactivity cutoff, via `LAG()` plus a running `SUM()` of new-session flags - one of the most reusable window-function idioms in analytics SQL.

In [22]:
sessions = run("""
WITH ordered_events AS (
    SELECT user_id, event_timestamp, event_name, device,
           LAG(event_timestamp) OVER (PARTITION BY user_id ORDER BY event_timestamp) AS prior_event_time
    FROM website_events
),
session_flags AS (
    SELECT *,
           CASE WHEN prior_event_time IS NULL
                     OR (julianday(event_timestamp) - julianday(prior_event_time)) * 24 * 60 > 30
                THEN 1 ELSE 0 END AS is_new_session
    FROM ordered_events
),
sessions AS (
    SELECT *, SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_timestamp) AS session_number
    FROM session_flags
)
SELECT user_id, session_number,
       MIN(event_timestamp) AS session_start,
       MAX(event_timestamp) AS session_end,
       COUNT(*) AS events_in_session,
       device
FROM sessions
GROUP BY user_id, session_number
ORDER BY user_id, session_number
LIMIT 15;
""")
display(sessions)

,user_id,session_number,session_start,session_end,events_in_session,device
0,1,1,2025-02-19 15:00:00,2025-02-19 15:12:00,5,desktop
1,1,2,2025-05-17 19:00:00,2025-05-17 19:00:00,1,desktop
2,1,3,2025-08-12 13:00:00,2025-08-12 13:03:00,2,desktop
3,1,4,2025-11-26 15:00:00,2025-11-26 15:00:00,1,mobile
4,1,5,2026-04-26 12:00:00,2026-04-26 12:03:00,2,mobile
5,2,1,2025-07-10 13:00:00,2025-07-10 13:00:00,1,mobile
6,2,2,2025-08-31 16:00:00,2025-08-31 16:06:00,3,desktop
7,3,1,2024-04-18 18:00:00,2024-04-18 18:03:00,2,mobile
8,3,2,2025-04-09 17:00:00,2025-04-09 17:06:00,3,desktop
9,3,3,2025-07-14 10:00:00,2025-07-14 10:00:00,1,desktop


## 14. Funnel Analysis (Parts 20, 35)

Step-by-step conversion through visited -> viewed_product -> added_to_cart -> started_checkout -> purchased.

In [23]:
funnel = run("""
WITH steps AS (
    SELECT user_id,
           MAX(CASE WHEN event_name='visited' THEN 1 ELSE 0 END) AS s1_visited,
           MAX(CASE WHEN event_name='viewed_product' THEN 1 ELSE 0 END) AS s2_viewed,
           MAX(CASE WHEN event_name='added_to_cart' THEN 1 ELSE 0 END) AS s3_cart,
           MAX(CASE WHEN event_name='started_checkout' THEN 1 ELSE 0 END) AS s4_checkout,
           MAX(CASE WHEN event_name='purchased' THEN 1 ELSE 0 END) AS s5_purchased
    FROM website_events GROUP BY user_id
)
SELECT SUM(s1_visited) AS visited, SUM(s2_viewed) AS viewed_product, SUM(s3_cart) AS added_to_cart,
       SUM(s4_checkout) AS started_checkout, SUM(s5_purchased) AS purchased
FROM steps;
""")
display(funnel)

steps_labels = ['visited','viewed_product','added_to_cart','started_checkout','purchased']
values = funnel.iloc[0].tolist()
print("\nStep-to-step conversion:")
for i in range(1, len(values)):
    pct = (values[i] / values[i-1] * 100) if values[i-1] else 0
    print(f"  {steps_labels[i-1]} -> {steps_labels[i]}: {pct:.1f}%")


Step-to-step conversion:
  visited -> viewed_product: 92.6%
  viewed_product -> added_to_cart: 80.9%
  added_to_cart -> started_checkout: 70.3%
  started_checkout -> purchased: 50.8%


,visited,viewed_product,added_to_cart,started_checkout,purchased
0,243,225,182,128,65


## 15. Data Cleaning Demonstration (Part 6)

A quick before/after example: standardizing free-text channel values into controlled categories, and safely handling a divide-by-zero in a rate calculation.

In [24]:
before = run("SELECT channel, COUNT(*) AS n FROM orders GROUP BY channel;")
print("Channel values as stored (already fairly clean in this synthetic dataset):")
display(before)

cleaned = run("""
SELECT
    CASE
        WHEN LOWER(TRIM(channel)) IN ('online','web','website') THEN 'Online'
        WHEN LOWER(TRIM(channel)) IN ('mobile app','app','mobile') THEN 'Mobile App'
        WHEN LOWER(TRIM(channel)) IN ('in-store','store','retail') THEN 'In-Store'
        ELSE 'Other'
    END AS channel_clean,
    COUNT(*) AS n
FROM orders GROUP BY channel_clean ORDER BY n DESC;
""")
print("After standardization:")
display(cleaned)

Channel values as stored (already fairly clean in this synthetic dataset):
After standardization:


,channel,n
0,In-Store,165
1,Mobile App,321
2,Online,636


,channel_clean,n
0,Online,636
1,Mobile App,321
2,In-Store,165


## 16. SQL for Machine Learning: A Leakage-Safe Churn Feature Table (Parts 45-46)

Every feature is computed strictly **before** a cutoff date; the churn label is computed strictly **after** it, using a 90-day forward window. This is the exact pattern that should feed a `scikit-learn` churn model in Python - see Part 49 for the full SQL + Python workflow.

In [25]:
cutoff = '2026-03-01'

ml_table = run(f"""
WITH customer_orders AS (
    SELECT o.customer_id,
           COUNT(DISTINCT o.order_id) AS total_orders,
           SUM(oi.quantity*oi.unit_price*(1-oi.discount)) AS total_spend,
           AVG(oi.quantity*oi.unit_price*(1-oi.discount)) AS avg_order_value,
           MAX(o.order_date) AS last_order_date
    FROM orders o JOIN order_items oi ON oi.order_id = o.order_id
    WHERE o.order_date < '{cutoff}' AND o.order_status = 'completed'
    GROUP BY o.customer_id
),
support AS (
    SELECT customer_id, COUNT(*) AS support_tickets
    FROM support_tickets
    WHERE opened_date < '{cutoff}'
    GROUP BY customer_id
),
label AS (
    SELECT c.customer_id,
           CASE WHEN NOT EXISTS (
               SELECT 1 FROM orders o2
               WHERE o2.customer_id = c.customer_id
                 AND o2.order_status = 'completed'
                 AND o2.order_date >= '{cutoff}'
                 AND o2.order_date < date('{cutoff}', '+90 days')
           ) THEN 1 ELSE 0 END AS churn_label
    FROM customers c
)
SELECT co.customer_id,
       co.total_orders,
       ROUND(co.total_spend, 2) AS total_spend,
       ROUND(co.avg_order_value, 2) AS avg_order_value,
       CAST(julianday('{cutoff}') - julianday(co.last_order_date) AS INTEGER) AS days_since_last_order,
       COALESCE(sp.support_tickets, 0) AS support_tickets,
       l.churn_label
FROM customer_orders co
LEFT JOIN support sp ON sp.customer_id = co.customer_id
JOIN label l ON l.customer_id = co.customer_id;
""")

print(f"Feature table shape: {ml_table.shape}")
print(f"Churn rate in this feature/label table: {ml_table['churn_label'].mean()*100:.1f}%")
display(ml_table.head(10))

Feature table shape: (297, 7)
Churn rate in this feature/label table: 91.6%


,customer_id,total_orders,total_spend,avg_order_value,days_since_last_order,support_tickets,churn_label
0,3,2,2189.39,547.35,629,0,1
1,4,2,949.76,474.88,38,0,0
2,7,1,969.11,323.04,179,0,1
3,9,3,3805.84,543.69,355,0,1
4,10,2,2481.76,413.63,381,2,1
5,12,1,1381.74,690.87,501,0,1
6,13,3,2750.07,458.35,267,0,1
7,14,2,1696.41,565.47,422,0,1
8,15,4,8203.41,683.62,488,0,1
9,16,8,4881.25,287.13,156,0,1


In [26]:
# A quick sanity check that no leakage exists: every feature's underlying orders end before the cutoff
leak_check = run(f"""
SELECT MAX(order_date) AS latest_feature_order_date FROM orders
WHERE order_date < '{cutoff}' AND order_status = 'completed';
""")
display(leak_check)
assert leak_check.iloc[0,0] < cutoff, "Leakage check failed: a feature used data on/after the cutoff date."
print("Leakage check passed: all feature-window orders are strictly before the cutoff date.")

Leakage check passed: all feature-window orders are strictly before the cutoff date.


,latest_feature_order_date
0,2026-02-28


## 17. Summary

This workbook executed real, runnable SQL for the core techniques of the teaching note: KPI development, sales and profitability analytics, RFM segmentation, cohort retention, churn analysis, market basket analysis, z-score outlier detection, Pareto analysis, session construction, funnel analysis, data cleaning, and a leakage-safe machine learning feature table.

Every query pattern here generalizes directly to PostgreSQL, MySQL, SQL Server, Oracle, and cloud warehouses (Snowflake, BigQuery, Redshift) with the dialect adjustments noted throughout the main teaching note - most commonly, replacing `strftime()` and `julianday()` (SQLite-specific) with `DATE_TRUNC()`/`EXTRACT()` and native date arithmetic, and using native `STDDEV()`/`PERCENTILE_CONT()` where the engine provides them.

**Kajola Gbenga**
*Data Scientist & AI Engineer*